# Set up Tuesday's batches (week 3, day 2)

For the trainer. Run this notebook once in the class workspace before Tuesday, after Monday's setup, from your Git folder of the course: `week-03/setup/day-02-setup.ipynb`. The numbered steps, the Genie Agent's steps, and what to check are in `day-02-setup.md` beside it.

It makes three tables in `course.returns`: `batch_clean` (30 rows), `batch_dirty` (32 rows, the faults put in) and `batch_governed` (a copy of `batch_dirty` with a column mask on `tin` and a row filter by tax year). The rows and the SQL are in `day-02-course-data.py` beside this notebook. The course made up every row. Running it again is safe: it makes each table again.

Each code cell has a number on its first line. Run cells 1 to 5 in order.

In [ ]:
# 1. Settings. CLASS_GROUP holds the learners. REVIEWER_GROUP is an account group that holds you
#    and no learner: it sees every row and every identification number of batch_governed.
CLASS_GROUP = "account users"
REVIEWER_GROUP = ""  # for example week3_reviewers; make it in the account console first

import os
try:
    _path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _folder = os.path.dirname(_path if _path.startswith("/Workspace") else "/Workspace" + _path)
except NameError:  # outside Databricks: the folder the notebook runs in
    _folder = os.getcwd()
course = {"__name__": "course_data"}
exec(open(os.path.join(_folder, "day-02-course-data.py"), encoding="utf-8").read(), course)
assert REVIEWER_GROUP, "Set REVIEWER_GROUP: an account group that holds you and no learner."
print("The batches go in course.returns. Reviewers:", REVIEWER_GROUP, "| learners:", CLASS_GROUP)

In [ ]:
# 2. The clean batch and the dirty batch, with their column comments.
for statement in course["batch_statements"]():
    spark.sql(statement)
print("Made course.returns.batch_clean and course.returns.batch_dirty.")
for kind, what, rows in course["FAULTS"]:
    print(f"  dirty, {kind}: {what} ({', '.join(rows)})")

In [ ]:
# 3. batch_governed: a copy of the dirty batch, then the row filter and the column mask.
#    If the workspace or the compute refuses them, the copy stays plain: tell the class that
#    exercise 3 takes its written fallback.
statements = course["governed_statements"](REVIEWER_GROUP)
for statement in statements["copy"]:
    spark.sql(statement)
try:
    for statement in statements["controls"]:
        spark.sql(statement)
    print("Made course.returns.batch_governed with its row filter and its column mask.")
except Exception as refused:
    print("FALLBACK: no filter or mask on course.returns.batch_governed.", str(refused).splitlines()[0][:200])

In [ ]:
# 4. Read back. Expected: batch_clean 30 rows, batch_dirty 32 rows. batch_governed, as you
#    (a reviewer): 2019: 2, 2021: 8, 2022: 12, 2023: 10, every tin in full. A learner sees
#    2023: 10 and tin as ***-**-****.
for table in ["batch_clean", "batch_dirty"]:
    print(table, spark.sql(f"SELECT COUNT(*) AS n FROM course.returns.{table}").first()["n"], "rows")
years = spark.sql("SELECT tax_year, COUNT(*) AS n FROM course.returns.batch_governed GROUP BY tax_year ORDER BY tax_year").collect()
print("batch_governed, as you:", ", ".join(f"{r['tax_year']}: {r['n']}" for r in years))
print("first tin, as you:", spark.sql("SELECT tin FROM course.returns.batch_governed ORDER BY return_id LIMIT 1").first()["tin"])

In [ ]:
# 5. Read access for the class. Monday's cell 5 granted SELECT on the schema course.returns,
#    which reaches these tables too. These grants name each table, so SHOW GRANTS lists them.
for table in ["batch_clean", "batch_dirty", "batch_governed"]:
    spark.sql(f"GRANT SELECT ON TABLE course.returns.{table} TO `{CLASS_GROUP}`")
print("Granted SELECT on the three batches to", CLASS_GROUP)